# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/manahilrubabsatti/flyrank-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd
import numpy as np

# Load the same starter dataset used in Week 1 and Week 2
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("FlyRank ML-03 setup complete")
print("Working directory:", os.getcwd())
print("Dataset shape:", df.shape)
print("One row represents one page.")

FlyRank ML-03 setup complete
Working directory: /content/flyrank-ml-internship-starter
Dataset shape: (30000, 44)
One row represents one page.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

### My lane: Ranking

I am framing my Refresh / Content Opportunity Scoring lane as a **ranking task**. The goal is not simply to label pages as good or bad, but to rank pages so that a reviewer can see the pages that may be worth investigating first. Ranking fits the decision because the available review time is limited and the output needs to produce a prioritized list.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("ML task type: Ranking")
print("Decision: Which pages should be reviewed first?")
print("Unit of analysis: One page")


ML task type: Ranking
Decision: Which pages should be reviewed first?
Unit of analysis: One page


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target / proxy

I will use **declining trend** as a proxy for pages that may deserve review. The label is defined from the `trend_direction` column: a page is labeled 1 when its trend is `down`, and 0 otherwise. This is a defined proxy rather than a direct measurement of whether a page actually needs a refresh.


In [6]:
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down").astype(int)
)

print("Target column created: is_declining_label")
print("Total pages:", len(df))
print("Declining pages:", df["is_declining_label"].sum())
print("Declining rate:", round(df["is_declining_label"].mean(), 3))

Target column created: is_declining_label
Total pages: 30000
Declining pages: 16262
Declining rate: 0.542


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

### Success metric: Precision@50

I will use Precision@50 as the main success metric. It measures the proportion of the 50 highest-ranked pages that are actually declining according to the defined proxy. A higher Precision@50 means that the short review list contains more pages matching the target condition. This metric fits the decision because a reviewer has limited time and needs a useful prioritized list.

In [7]:
import numpy as np

def precision_at_k(scores, labels, k=50):
    scores = np.asarray(scores)
    labels = np.asarray(labels)

    order = np.argsort(-scores)
    top_k = labels[order[:k]]

    return top_k.mean()

print("Success metric: Precision@50")
print("Higher value = more declining pages in the top 50.")

Success metric: Precision@50
Higher value = more declining pages in the top 50.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

### Unit of analysis: one page

The unit of analysis is one page. Each row in the dataset represents one page and contains observable search and content signals such as impressions, average position, CTR, word count, content age, and update age. These page-level observations will be used to rank pages for possible review.

In [8]:
page_features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
    "trend_direction",
    "is_declining_label"
]

df[page_features].head(10)

,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,trend_direction,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,down,1
1,445,25,15320,20.3,0.05,2481.0,down,1
2,141,20,12581,36.5,0.09,3515.0,down,1
3,463,22,11751,6.2,0.49,NaN,stable,0
4,263,14,19140,44.0,0.13,2803.0,down,1
5,147,20,3970,8.5,0.03,3080.0,down,1
6,90,20,20,7.0,0.00,3059.0,down,1
7,445,22,1724,21.2,0.06,NaN,stable,0
8,90,20,32574,46.0,0.09,3807.0,down,1
9,257,104,1240,4.9,0.16,NaN,down,1


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A fixed rule such as prioritizing pages that are both stale and visible is useful as a simple baseline, but it requires manually choosing thresholds and deciding which signals matter. Page performance can involve several signals at the same time, and their relationships may be difficult to capture with one if-statement. ML can combine multiple observable signals and learn patterns that may be difficult to specify manually. I will still compare the ML approach with a simple rule rather than assuming that ML is automatically better.

In [9]:
stale = df["days_since_last_update"] >= 180
visible = df["impressions_90d"] >= 500

df["hand_rule_score"] = (
    stale.astype(int)
    * visible.astype(int)
    * df["impressions_90d"]
)

print("Pages selected by the stale + visible rule:", (df["hand_rule_score"] > 0).sum())

print("\nTop 5 pages from the simple rule:")
print(
    df.sort_values("hand_rule_score", ascending=False)[
        [
            "impressions_90d",
            "days_since_last_update",
            "avg_position",
            "ctr",
            "trend_direction",
            "hand_rule_score"
        ]
    ].head()
)

Pages selected by the stale + visible rule: 17

Top 5 pages from the simple rule:
       impressions_90d  days_since_last_update  avg_position   ctr  \
16751            61678                     194          19.7  0.15   
16514            59472                     194          24.8  0.13   
7021             25715                     194          22.2  0.23   
21268            13299                     193          10.5  0.49   
11489             7812                     194          39.0  0.01   

      trend_direction  hand_rule_score  
16751            down            61678  
16514            down            59472  
7021             down            25715  
21268            down            13299  
11489            down             7812  


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.